In [1]:
import datetime
import calendar
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
import datetime

In [2]:
# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
                    SELECT * FROM CUN.MALLASLAURA
             ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        mallascun = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()

Conexion a la base de datos realizada


In [3]:
df = mallascun.copy()
df.shape

(1729, 46)

In [4]:
# ─── 2. COLUMNAS COMUNES A TODOS LOS NIVELES ──────────────────────────────────
COLS_COMUNES = ['ESTADO', 'ESCUELAS', 'SEDE', 'MODALIDAD', 'PENSUM', 'IDIOMAS']

# Créditos totales de la cadena (suma de los 3 niveles por fila)
def creditos_totales(row):
    vals = [
        pd.to_numeric(row.get('TOTAL_CREDITOS_TECNICO'),   errors='coerce'),
        pd.to_numeric(row.get('TOTAL_CREDITOS_TECNOLOGO'), errors='coerce'),
        pd.to_numeric(str(row.get('TOTAL_CREDITOS_PROFESIONAL', '')).split()[0], errors='coerce'),
    ]
    return np.nansum([v for v in vals if not np.isnan(v)]) or np.nan

df['_TOTAL_CREDITOS_CADENA'] = df.apply(creditos_totales, axis=1)
print('Créditos totales de cadena calculados ✓')




Créditos totales de cadena calculados ✓


In [5]:
# ─── 3. FUNCIÓN AUXILIAR DE UNPIVOT POR NIVEL ─────────────────────────────────
def extraer_nivel(df_src, nivel,
                  col_cod, col_snies, col_res,
                  col_fi, col_ff,
                  col_nombre, col_titulo,
                  col_semestres, col_creditos_nivel):
    """
    Filtra las filas que tienen datos para el nivel dado y devuelve
    un DataFrame con la estructura de MALLAS CON.
    """
    mask = df_src[col_cod].notna()
    sub  = df_src[mask].copy()

    out = pd.DataFrame({
        'ESTADO'               : sub['ESTADO'],
        'ESCUELAS'             : sub['ESCUELAS'],
        'COD_PROGRAMA'         : sub[col_cod],
        'SNIES'                : sub[col_snies],
        'RESOLUCION'           : sub[col_res],
        'SEDE'                 : sub['SEDE'],
        'MODALIDAD'            : sub['MODALIDAD'],
        'FECHA_INICIO'         : pd.to_datetime(sub[col_fi],  errors='coerce'),
        'FECHA_FINAL'          : pd.to_datetime(sub[col_ff],  errors='coerce'),
        'NOMBRE_PROGRAMA'      : sub[col_nombre],
        'PENSUM'               : sub['PENSUM'],
        'SEMESTRES'            : pd.to_numeric(sub[col_semestres], errors='coerce'),
        'TOTAL_CREDITOS_NIVEL' : pd.to_numeric(sub[col_creditos_nivel], errors='coerce'),
        'TITULACION'           : sub[col_titulo],
        'TOTAL_CREDITOS'       : sub['_TOTAL_CREDITOS_CADENA'],
        'NIVEL'                : nivel,
    })
    return out

In [6]:
# ─── 4. GENERAR FILAS POR NIVEL ───────────────────────────────────────────────

# --- TÉCNICO ---
df_tecnico = extraer_nivel(
    df, nivel='TECNICO',
    col_cod            = 'COD_PROG_TECNICO',
    col_snies          = 'CODIGO_SNIES_TECNICO',
    col_res            = 'RESOLUCION_TECNICO',
    col_fi             = 'FECHA_INICIO_TEC',
    col_ff             = 'FECHA_FIN_TEC',
    col_nombre         = 'NOMBRE_DEL_PROGRAMA_TEC',
    col_titulo         = 'TITULO_OTORGADO_TEC',
    col_semestres      = 'TECNICO_NIVELES',
    col_creditos_nivel = 'TOTAL_CREDITOS_TECNICO',
)

# --- TECNÓLOGO ---
df_tecnologo = extraer_nivel(
    df, nivel='TECNOLOGO',
    col_cod            = 'COD_PROG_TECNOLOGO',
    col_snies          = 'CODIGO_SNIES_TECNOLOGICO',
    col_res            = 'RESOLUCION_TNG',
    col_fi             = 'FECHA_INICIO_TNG',
    col_ff             = 'FECHA_FIN_TNG',
    col_nombre         = 'NOMBRE_DEL_PROGRAMA_TNG',
    col_titulo         = 'TITULO_OTORGADO_TNG',
    col_semestres      = 'TECNOLOGO_NIVELES',
    col_creditos_nivel = 'TOTAL_CREDITOS_TECNOLOGO',
)

# --- PROFESIONAL / ESPECIALIZACIÓN ---
# Filas cuyo PENSUM empieza con 'ESP' se clasifican como ESPECIALIZACIÓN
df['_es_esp'] = df['PENSUM'].astype(str).str.startswith('ESP')

df_profesional = extraer_nivel(
    df[df['COD_PROG_PROFESIONAL'].notna() & ~df['_es_esp']], nivel='PROFESIONAL',
    col_cod            = 'COD_PROG_PROFESIONAL',
    col_snies          = 'CODIGO_SNIES_PROF',
    col_res            = 'RESOLUCION_PROF',
    col_fi             = 'FECHA_INICIO_PRO',
    col_ff             = 'FECHA_FIN_PRO',
    col_nombre         = 'NOMBRE_DEL_PROGRAMA_PRO',
    col_titulo         = 'TITULO_OTORGADO_PRO',
    col_semestres      = 'PROFESIONAL_NIVELES',
    col_creditos_nivel = 'TOTAL_CREDITOS_PROFESIONAL',
)

df_especializacion = extraer_nivel(
    df[df['COD_PROG_PROFESIONAL'].notna() & df['_es_esp']], nivel='ESPECIALIZACION',
    col_cod            = 'COD_PROG_PROFESIONAL',
    col_snies          = 'CODIGO_SNIES_PROF',
    col_res            = 'RESOLUCION_PROF',
    col_fi             = 'FECHA_INICIO_PRO',
    col_ff             = 'FECHA_FIN_PRO',
    col_nombre         = 'NOMBRE_DEL_PROGRAMA_PRO',
    col_titulo         = 'TITULO_OTORGADO_PRO',
    col_semestres      = 'PROFESIONAL_NIVELES',
    col_creditos_nivel = 'TOTAL_CREDITOS_PROFESIONAL',
)

print(f'TÉCNICO       : {len(df_tecnico):>5} filas')
print(f'TECNÓLOGO     : {len(df_tecnologo):>5} filas')
print(f'PROFESIONAL   : {len(df_profesional):>5} filas')
print(f'ESPECIALIZAC. : {len(df_especializacion):>5} filas')

TÉCNICO       :  1729 filas
TECNÓLOGO     :  1729 filas
PROFESIONAL   :  1700 filas
ESPECIALIZAC. :    29 filas


In [7]:
# ─── 5. CONSOLIDAR Y ORDENAR ──────────────────────────────────────────────────
ORDEN_NIVEL = ['TECNICO', 'TECNOLOGO', 'PROFESIONAL', 'ESPECIALIZACION']

df_resultado = (
    pd.concat([df_tecnico, df_tecnologo, df_profesional, df_especializacion],
              ignore_index=True)
    .assign(NIVEL=lambda x: pd.Categorical(x['NIVEL'], categories=ORDEN_NIVEL, ordered=True))
    .sort_values(['ESCUELAS', 'PENSUM', 'NIVEL'])
    .reset_index(drop=True)
)

# Limpiar columna temporal de la fuente
df.drop(columns=['_TOTAL_CREDITOS_CADENA', '_es_esp'], inplace=True)

print(f'Total filas resultado: {len(df_resultado)}')
print(f'Columnas: {df_resultado.columns.tolist()}')
df_resultado.head(10)

Total filas resultado: 5187
Columnas: ['ESTADO', 'ESCUELAS', 'COD_PROGRAMA', 'SNIES', 'RESOLUCION', 'SEDE', 'MODALIDAD', 'FECHA_INICIO', 'FECHA_FINAL', 'NOMBRE_PROGRAMA', 'PENSUM', 'SEMESTRES', 'TOTAL_CREDITOS_NIVEL', 'TITULACION', 'TOTAL_CREDITOS', 'NIVEL']


,ESTADO,ESCUELAS,COD_PROGRAMA,SNIES,RESOLUCION,SEDE,MODALIDAD,FECHA_INICIO,FECHA_FINAL,NOMBRE_PROGRAMA,PENSUM,SEMESTRES,TOTAL_CREDITOS_NIVEL,TITULACION,TOTAL_CREDITOS,NIVEL
0,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TVC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
1,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TAM04,8646,1833,ARMENIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
2,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
3,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBV04,8646,1833,BUENAVENTURA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
4,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TCA03,8646,1833,CERES ALBANIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
5,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TDO04,8646,1833,CERES DONCELLO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
6,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TMQ04,8646,1833,CERES MARIQUITA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
7,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TCH02,8646,1833,CHIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,126.0,TECNICO
8,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TAP03,8646,1833,CUNAD APARTADO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,126.0,TECNICO
9,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TCR04,8646,1833,CUNAD CARMEN DE APICALA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO


In [8]:
# ─── 6. VALIDACIÓN RÁPIDA ─────────────────────────────────────────────────────
print('Conteo por NIVEL:')
print(df_resultado['NIVEL'].value_counts().to_string())
print()
print('Nulos por columna:')
print(df_resultado.isnull().sum().to_string())
print()
print('Muestra aleatoria:')
df_resultado.sample(5, random_state=42)

Conteo por NIVEL:
NIVEL
TECNICO            1729
TECNOLOGO          1729
PROFESIONAL        1700
ESPECIALIZACION      29

Nulos por columna:
ESTADO                     0
ESCUELAS                   0
COD_PROGRAMA               0
SNIES                      0
RESOLUCION                 0
SEDE                       0
MODALIDAD                  0
FECHA_INICIO            1873
FECHA_FINAL             1876
NOMBRE_PROGRAMA            0
PENSUM                     0
SEMESTRES               1587
TOTAL_CREDITOS_NIVEL    1596
TITULACION                 0
TOTAL_CREDITOS           330
NIVEL                      0

Muestra aleatoria:


,ESTADO,ESCUELAS,COD_PROGRAMA,SNIES,RESOLUCION,SEDE,MODALIDAD,FECHA_INICIO,FECHA_FINAL,NOMBRE_PROGRAMA,PENSUM,SEMESTRES,TOTAL_CREDITOS_NIVEL,TITULACION,TOTAL_CREDITOS,NIVEL
3362,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,GLB04,53142,6482,CUNAD LIBANO,DISTANCIA,2007-10-26,2014-10-24,TECNOLOGIA EN GESTION CONTABLE Y FINANCIERA,1064,6.0,63.0,TECNOLOGO EN GESTION CONTABLE Y FINANCIERA,210.0,TECNOLOGO
4671,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA,PRESENCIAL,NaT,NaT,N/A,8014,NaN,NaN,N/A,57.0,PROFESIONAL
3542,VENCIDO,ESCUELA DE TRANSFORMACION EMPRESARIAL,GPA02,53142,2143,CUNAD PAICOL,DISTANCIA,2011-03-18,2018-03-16,TECNOLOGIA EN GESTION CONTABLE Y FINANCIERA,1067,NaN,NaN,TECNOLOGO EN GESTION CONTABLE Y FINANCIERA,59.0,TECNOLOGO
472,VENCIDO,"ESCUELA DE CIENCIAS SOCIALES, JURIDICAS Y GOBI...",GTO05,53577,nan,CAD - TUMACO,DISTANCIA,NaT,NaT,TECNOLOGIA EN GESTION PUBLICA TERRITORIAL,1029,6.0,36.0,TECNOLOGO EN GESTION PUBLICA TERRITORIAL,174.0,TECNOLOGO
1788,VIGENTE,ESCUELA DE INGENIERIAS,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPTI,NaN,NaN,N/A,28.0,TECNOLOGO


In [9]:
# ─── 7. EXPORTAR A EXCEL ──────────────────────────────────────────────────────
df_resultado = df_resultado.drop_duplicates()
df_resultado

,ESTADO,ESCUELAS,COD_PROGRAMA,SNIES,RESOLUCION,SEDE,MODALIDAD,FECHA_INICIO,FECHA_FINAL,NOMBRE_PROGRAMA,PENSUM,SEMESTRES,TOTAL_CREDITOS_NIVEL,TITULACION,TOTAL_CREDITOS,NIVEL
0,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TVC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
1,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TAM04,8646,1833,ARMENIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
2,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
3,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBV04,8646,1833,BUENAVENTURA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
4,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TCA03,8646,1833,CERES ALBANIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5182,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPPE,NaN,NaN,N/A,28.0,TECNOLOGO
5183,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESP17,117816,001807,BOGOTA CENTRO,VIRTUAL,2025-02-10,2032-02-09,ESPECIALIZACIÓN EN PROSPECTIVA ESTRATÉGICA,ESPPE,NaN,28.0,ESPECIALISTA EN PROSPECTIVA ESTRATÉGICA,28.0,ESPECIALIZACION
5184,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPSS,NaN,NaN,N/A,28.0,TECNICO
5185,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPSS,NaN,NaN,N/A,28.0,TECNOLOGO


In [10]:
for col in df_resultado.select_dtypes(include=['object']).columns:
    df_resultado[col] = df_resultado[col].fillna('')
df_resultado['TOTAL_CREDITOS'] = df_resultado['TOTAL_CREDITOS'].replace('', 0)
df_resultado.shape

(4531, 16)

In [11]:
for col in df_resultado.select_dtypes(include=['object', 'string']).columns:
    df_resultado[col] = (
        df_resultado[col]
        .astype(str)
        .str.replace(r'\.0$', '', regex=True)
        .str.strip()
    )
df_resultado = df_resultado.applymap(lambda x: x.strip() if isinstance(x, str) else x)   
df_resultado

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_22564\1522469466.py:8: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_resultado = df_resultado.applymap(lambda x: x.strip() if isinstance(x, str) else x)


,ESTADO,ESCUELAS,COD_PROGRAMA,SNIES,RESOLUCION,SEDE,MODALIDAD,FECHA_INICIO,FECHA_FINAL,NOMBRE_PROGRAMA,PENSUM,SEMESTRES,TOTAL_CREDITOS_NIVEL,TITULACION,TOTAL_CREDITOS,NIVEL
0,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TVC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
1,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TAM04,8646,1833,ARMENIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
2,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
3,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBV04,8646,1833,BUENAVENTURA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
4,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TCA03,8646,1833,CERES ALBANIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4.0,84.0,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210.0,TECNICO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5182,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPPE,NaN,NaN,N/A,28.0,TECNOLOGO
5183,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESP17,117816,001807,BOGOTA CENTRO,VIRTUAL,2025-02-10,2032-02-09,ESPECIALIZACIÓN EN PROSPECTIVA ESTRATÉGICA,ESPPE,NaN,28.0,ESPECIALISTA EN PROSPECTIVA ESTRATÉGICA,28.0,ESPECIALIZACION
5184,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPSS,NaN,NaN,N/A,28.0,TECNICO
5185,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPSS,NaN,NaN,N/A,28.0,TECNOLOGO


In [12]:
def quitar_decimal_cero(df_resultado):
    """
    Elimina el .0 de todas las columnas numéricas del DataFrame
    de forma dinámica.
    
    """

    for col in df_resultado.columns:

        df_resultado[col] = df_resultado[col].apply(
            lambda x:
                str(int(float(x)))
                if (
                    pd.notnull(x)
                    and str(x).replace('.', '', 1).isdigit()
                    and float(x).is_integer()
                )
                else x
        )

    return df_resultado
df_resultado = quitar_decimal_cero(df_resultado)

In [13]:
df_resultado['TOTAL_CREDITOS'] = df_resultado['TOTAL_CREDITOS'].fillna(0)
df_resultado

,ESTADO,ESCUELAS,COD_PROGRAMA,SNIES,RESOLUCION,SEDE,MODALIDAD,FECHA_INICIO,FECHA_FINAL,NOMBRE_PROGRAMA,PENSUM,SEMESTRES,TOTAL_CREDITOS_NIVEL,TITULACION,TOTAL_CREDITOS,NIVEL
0,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TVC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4,84,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210,TECNICO
1,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TAM04,8646,1833,ARMENIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4,84,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210,TECNICO
2,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBC05,8646,1833,BOGOTA CENTRO,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4,84,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210,TECNICO
3,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TBV04,8646,1833,BUENAVENTURA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4,84,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210,TECNICO
4,VENCIDO,ESCUELA DE CIENCIAS JURIDICAS Y GOBIERNO,TCA03,8646,1833,CERES ALBANIA,DISTANCIA,2008-04-09,2015-04-08,TECNICA PROFESIONAL EN PROCESOS ADMINISTRATIVO...,1021,4,84,TECNICO PROFESIONAL EN PROCESOS ADMINISTRATIVO...,210,TECNICO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5182,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPPE,NaN,NaN,N/A,28,TECNOLOGO
5183,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,ESP17,117816,1807,BOGOTA CENTRO,VIRTUAL,2025-02-10,2032-02-09,ESPECIALIZACIÓN EN PROSPECTIVA ESTRATÉGICA,ESPPE,NaN,28,ESPECIALISTA EN PROSPECTIVA ESTRATÉGICA,28,ESPECIALIZACION
5184,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPSS,NaN,NaN,N/A,28,TECNICO
5185,VIGENTE,ESCUELA DE TRANSFORMACION EMPRESARIAL,N/A,N/A,N/A,BOGOTA CENTRO,VIRTUAL,NaT,NaT,N/A,ESPSS,NaN,NaN,N/A,28,TECNOLOGO


In [14]:
df_final = df_resultado[df_resultado['COD_PROGRAMA'] != 'nan'].copy()
df_final.shape

(4048, 16)

In [15]:
#from datetime import datetime
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print("Error:", e)

Conexion a la base de datos realizada


In [16]:
with engine.connect() as connection:
    try:
        # 
        sql_query = text(f''' 
                    SELECT COLUMN_NAME, DATA_TYPE
                    FROM INFORMATION_SCHEMA.COLUMNS
                    WHERE TABLE_SCHEMA = 'CUN'
                    AND TABLE_NAME  = 'MALLASCUN'
                        ''')

        # CONVIERTE RESULTADO DE CONSULTA EN DATAFRAME    
        df_mallas_datos = pd.read_sql_query(sql_query, engine)

    finally:
        
        connection.close()
df_mallas_datos.shape

(16, 2)

In [17]:
map_types = {
    'varchar': 'string',
    'nvarchar': 'string',
    'char': 'string',
    'text': 'string',
    
    'int': 'Int64',
    'bigint': 'Int64',
    'smallint': 'Int64',
    
    'decimal': 'float64',
    'numeric': 'float64',
    'float': 'float64',
    
    'date': 'datetime64[ns]',
    'datetime': 'datetime64[ns]',
    'datetime2': 'datetime64[ns]'
}

In [18]:
dict_types = {}

for _, row in df_mallas_datos.iterrows():
    col = row['COLUMN_NAME']
    sql_type = row['DATA_TYPE'].lower()
    
    if sql_type in map_types:
        dict_types[col] = map_types[sql_type]

In [19]:
for col, dtype in dict_types.items():
    if col in df_final.columns:
        try:
            if 'datetime' in dtype:
                df_final[col] = pd.to_datetime(df_final[col], errors='coerce')
            else:
                df_final[col] = df_final[col].astype(dtype)
        except Exception as e:
            print(f"Error en columna {col}: {e}")

In [20]:
from sqlalchemy import text

with engine.connect() as connection:
    trans = connection.begin()
    try:
        sql_query = text("""
            DELETE FROM CUN.MALLASCUN
        """)

        connection.execute(sql_query)
        trans.commit()

    except Exception as e:
        trans.rollback()
        print(e)

    finally:
        connection.close()

In [21]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'MALLASCUN'
schema_name = 'CUN'
# Send the data to sql server
df_final.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='append',
    index=False,
    chunksize = 10000
)

# End connection

C:\Users\alejandro_arboledas\AppData\Local\Temp\ipykernel_22564\724321172.py:14: UserWarning: The provided table name 'MALLASCUN' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df_final.to_sql(


-1